# Shield AML Classifier Training (No Download Needed)

**Purpose:** Train a RandomForest classifier to detect money laundering, then export to ONNX for the Java backend.

**No Kaggle account, no downloads, no external data needed.** The IBM AML dataset is itself synthetic (generated by a multi-agent virtual world model — see https://github.com/IBM/AML-Data). This notebook generates equivalent synthetic transactions with the same schema and the same laundering patterns (structuring, rapid transit, round-tripping, smurfing), then trains a real classifier on them.

**Output:** `aml_classifier.onnx` + `feature_schema.json` -> put in `src/main/resources/models/`

**No GPU needed.** Runs in ~2 min on free Colab.

In [ ]:
!pip install skl2onnx onnxruntime pandas scikit-learn numpy

## Step 1: Generate synthetic AML transaction data

Same column schema as the IBM dataset. We inject 4 laundering patterns: structuring/smurfing, rapid transit, round-tripping, and fan-out scattering.

In [ ]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)
random.seed(42)

N_ACCOUNTS = 2000      # total accounts in the virtual world
N_CLEAN = 60000        # clean transactions
N_LAUNDER = 4000       # laundering transactions (6.7% rate — higher than real, for training)

accounts = [f"acc_{i:05d}" for i in range(N_ACCOUNTS)]
banks = [f"bank_{i:02d}" for i in range(20)]
currencies = ['USD', 'EUR', 'GBP', 'SAR', 'AED']
payment_formats = ['ACH', 'WIRE', 'CHECK', 'CARD', 'CRYPTO', 'CASH']

rows = []

# --- Clean transactions (legitimate business activity) ---
for _ in range(N_CLEAN):
    sender = random.choice(accounts)
    receiver = random.choice([a for a in accounts if a != sender])
    amount = np.random.lognormal(mean=5.5, sigma=1.8)  # most transfers $50-$5000
    amount = round(min(amount, 50000), 2)
    rows.append({
        'time_step': random.randint(1, 100),
        'from_bank': random.choice(banks),
        'from_account': sender,
        'to_bank': random.choice(banks),
        'to_account': receiver,
        'amount_received': amount,
        'receiving_currency': 'USD',
        'amount_paid': amount,
        'payment_currency': 'USD',
        'payment_format': random.choice(['ACH', 'WIRE', 'CHECK', 'CARD']),
        'is_laundering': 0,
    })

# --- Laundering Pattern 1: Structuring / Smurfing ---
# One sender splits a large amount into many small sub-threshold transfers
for _ in range(800):
    sender = random.choice(accounts)
    target = random.choice(accounts)
    total = np.random.uniform(20000, 80000)
    n_splits = random.randint(4, 12)
    base_time = random.randint(1, 90)
    for j in range(n_splits):
        amt = round(total / n_splits * np.random.uniform(0.8, 1.2), 2)
        amt = min(amt, 9500)  # keep below $10k SAR threshold
        rows.append({
            'time_step': base_time + (j if j < 10 else 9),
            'from_bank': random.choice(banks),
            'from_account': sender,
            'to_bank': random.choice(banks),
            'to_account': target if random.random() > 0.3 else random.choice(accounts),
            'amount_received': amt, 'receiving_currency': 'USD',
            'amount_paid': amt, 'payment_currency': 'USD',
            'payment_format': random.choice(['ACH', 'WIRE', 'CASH']),
            'is_laundering': 1,
        })

# --- Laundering Pattern 2: Rapid Transit (pass-through / mule chain) ---
# Funds go A->B->C quickly, B is a mule that passes money through
for _ in range(600):
    a, b, c = random.sample(accounts, 3)
    amt = round(np.random.uniform(5000, 40000), 2)
    t = random.randint(1, 95)
    # A -> B
    rows.append({
        'time_step': t, 'from_bank': random.choice(banks), 'from_account': a,
        'to_bank': random.choice(banks), 'to_account': b,
        'amount_received': amt, 'receiving_currency': 'USD',
        'amount_paid': amt, 'payment_currency': 'USD',
        'payment_format': 'WIRE', 'is_laundering': 1,
    })
    # B -> C (same or next timestep, ~90% of the amount)
    rows.append({
        'time_step': t + 1, 'from_bank': random.choice(banks), 'from_account': b,
        'to_bank': random.choice(banks), 'to_account': c,
        'amount_received': round(amt * np.random.uniform(0.85, 0.98), 2),
        'receiving_currency': 'USD', 'amount_paid': round(amt * np.random.uniform(0.85, 0.98), 2),
        'payment_currency': 'USD', 'payment_format': 'WIRE', 'is_laundering': 1,
    })

# --- Laundering Pattern 3: Round-tripping (A->B->A) ---
for _ in range(400):
    a, b = random.sample(accounts, 2)
    amt = round(np.random.uniform(3000, 25000), 2)
    t = random.randint(1, 80)
    # A -> B
    rows.append({
        'time_step': t, 'from_bank': random.choice(banks), 'from_account': a,
        'to_bank': random.choice(banks), 'to_account': b,
        'amount_received': amt, 'receiving_currency': 'USD',
        'amount_paid': amt, 'payment_currency': 'USD',
        'payment_format': random.choice(['WIRE', 'CRYPTO']), 'is_laundering': 1,
    })
    # B -> A (money comes back, slightly less)
    rows.append({
        'time_step': t + random.randint(2, 7), 'from_bank': random.choice(banks), 'from_account': b,
        'to_bank': random.choice(banks), 'to_account': a,
        'amount_received': round(amt * 0.9, 2), 'receiving_currency': 'USD',
        'amount_paid': round(amt * 0.9, 2), 'payment_currency': 'USD',
        'payment_format': random.choice(['WIRE', 'CRYPTO']), 'is_laundering': 1,
    })

# --- Laundering Pattern 4: Fan-out scattering (one sender to many accounts) ---
for _ in range(400):
    sender = random.choice(accounts)
    amt = round(np.random.uniform(500, 5000), 2)
    t = random.randint(1, 98)
    n_targets = random.randint(5, 15)
    for _ in range(n_targets):
        rows.append({
            'time_step': t, 'from_bank': random.choice(banks), 'from_account': sender,
            'to_bank': random.choice(banks), 'to_account': random.choice(accounts),
            'amount_received': amt, 'receiving_currency': 'USD',
            'amount_paid': amt, 'payment_currency': random.choice(currencies),
            'payment_format': random.choice(['WIRE', 'CRYPTO', 'CASH']),
            'is_laundering': 1,
        })

df = pd.DataFrame(rows)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f'Total transactions: {len(df):,}')
print(f'Laundering cases: {df["is_laundering"].sum():,}')
print(f'Laundering rate: {df["is_laundering"].mean()*100:.2f}%')
print(f'Payment formats: {df["payment_format"].unique()}')
print()
print(df.head(10))

## Step 2: Feature engineering

Same features as the Java `AmlFeatures.java` — per-account stats that capture laundering signals.

In [ ]:
import numpy as np

print('Engineering account-level features...')

# Sender stats (from outgoing history)
sender_stats = df.groupby('from_account').agg(
    txn_count=('amount_paid', 'count'),
    total_sent=('amount_paid', 'sum'),
    avg_sent=('amount_paid', 'mean'),
    std_sent=('amount_paid', 'std'),
    min_sent=('amount_paid', 'min'),
    max_sent=('amount_paid', 'max'),
    unique_beneficiaries=('to_account', 'nunique'),
    active_days=('time_step', 'nunique'),
    payment_formats_used=('payment_format', 'nunique'),
).reset_index()
sender_stats['avg_daily_txns'] = sender_stats['txn_count'] / sender_stats['active_days'].clip(lower=1)
sender_stats['fan_out_ratio'] = sender_stats['unique_beneficiaries'] / sender_stats['txn_count'].clip(lower=1)
sender_stats['std_sent'] = sender_stats['std_sent'].fillna(0)

# Receiver stats (from incoming history)
receiver_stats = df.groupby('to_account').agg(
    recv_count=('amount_received', 'count'),
    total_received=('amount_received', 'sum'),
    avg_received=('amount_received', 'mean'),
    std_received=('amount_received', 'std'),
    unique_senders=('from_account', 'nunique'),
).reset_index()
receiver_stats['fan_in_ratio'] = receiver_stats['unique_senders'] / receiver_stats['recv_count'].clip(lower=1)
receiver_stats['std_received'] = receiver_stats['std_received'].fillna(0)

# Merge onto each transaction
feat_df = df.merge(sender_stats, on='from_account', how='left', suffixes=('', '_s'))
feat_df = feat_df.merge(receiver_stats, on='to_account', how='left', suffixes=('', '_r'))

# Structuring signal
SAR_THRESHOLD = 10000
feat_df['below_threshold'] = (feat_df['amount_paid'] < SAR_THRESHOLD).astype(int)
feat_df['amount_to_avg_ratio'] = feat_df['amount_paid'] / feat_df['avg_sent'].clip(lower=0.01)

# Rapid transit: does the receiver also send?
receiver_sender_overlap = set(df['from_account']) & set(df['to_account'])
feat_df['receiver_also_sends'] = feat_df['to_account'].isin(receiver_sender_overlap).astype(int)

# Currency mismatch
feat_df['currency_mismatch'] = (feat_df['payment_currency'] != feat_df['receiving_currency']).astype(int)

# Final feature columns — MUST match Java AmlFeatures.FEATURE_NAMES exactly
FEATURE_COLS = [
    'amount_paid', 'below_threshold', 'amount_to_avg_ratio',
    'txn_count', 'total_sent', 'avg_sent', 'std_sent', 'min_sent', 'max_sent',
    'unique_beneficiaries', 'active_days', 'avg_daily_txns', 'fan_out_ratio',
    'payment_formats_used',
    'recv_count', 'total_received', 'avg_received', 'std_received',
    'unique_senders', 'fan_in_ratio',
    'receiver_also_sends', 'currency_mismatch',
]

for c in FEATURE_COLS:
    feat_df[c] = pd.to_numeric(feat_df[c], errors='coerce').fillna(0)

X = feat_df[FEATURE_COLS].values.astype(np.float32)
y = feat_df['is_laundering'].values.astype(int)

print(f'Feature matrix shape: {X.shape}')
print(f'Features: {FEATURE_COLS}')
print(f'Positive class: {y.sum()} ({y.mean()*100:.2f}%)')

## Step 3: Train / test split (by time, no leakage)

In [ ]:
time_median = feat_df['time_step'].median()
train_mask = feat_df['time_step'] <= time_median
test_mask = feat_df['time_step'] > time_median

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Train: {len(X_train):,} ({y_train.sum()} laundering)')
print(f'Test:  {len(X_test):,} ({y_test.sum()} laundering)')

## Step 4: Train the RandomForest

No GPU needed. class_weight=balanced because laundering is ~6% of data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print('Training RandomForest...')
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)
print('Training complete.')

## Step 5: Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Clean', 'Laundering']))
print('=== Confusion Matrix ===')
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f'True Positives: {cm[1,1]}')
print(f'False Positives: {cm[0,1]}')
print(f'False Negatives: {cm[1,0]}')
try:
    auc = roc_auc_score(y_test, y_proba)
    print(f'ROC-AUC: {auc:.4f}')
except:
    pass

print()
print('=== Feature Importances ===')
importances = model.feature_importances_
for name, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: -x[1]):
    print(f'  {name}: {imp:.4f}')

## Step 6: Export to ONNX

This is the critical step — the ONNX model loads directly in Java via ONNX Runtime.

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import os, json

print('Exporting to ONNX...')
initial_type = [('features', FloatTensorType([None, len(FEATURE_COLS)]))]
onnx_model = convert_sklearn(model, initial_types=initial_type, target_opset=15)

onnx_path = 'aml_classifier.onnx'
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())
print(f'ONNX model saved: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

# Verify it runs
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
input_name = sess.get_inputs()[0].name
sample = X_test[:5].astype(np.float32)
pred = sess.run(None, {input_name: sample})
print('ONNX verification OK. Predictions:', pred[0].flatten())

## Step 7: Save feature schema

Java needs this to know the exact feature order.

In [ ]:
schema = {
    'features': FEATURE_COLS,
    'feature_count': len(FEATURE_COLS),
    'model_type': 'RandomForestClassifier',
    'sar_threshold': SAR_THRESHOLD,
    'training_rows': int(len(X_train)),
    'laundering_rate_train': float(y_train.mean()),
    'metrics': {
        'roc_auc': float(roc_auc_score(y_test, y_proba)) if len(set(y_test)) > 1 else None,
        'true_positives': int(cm[1, 1]),
        'false_positives': int(cm[0, 1]),
        'false_negatives': int(cm[1, 0]),
    },
}
schema_path = 'feature_schema.json'
with open(schema_path, 'w') as f:
    json.dump(schema, f, indent=2)
print(f'Feature schema saved: {schema_path}')
print(json.dumps(schema, indent=2))

## Step 8: Download both files

After downloading, place both in:
`mer9ad-backend-master/src/main/resources/models/`

Then build and run the backend. The AmlAgent will auto-load the ONNX model.

In [ ]:
from google.colab import files
files.download(onnx_path)
files.download(schema_path)